# Streamlit Movie Recommendation System

## Objective

In this notebook, we prepare all the components required for building an interactive Streamlit application.

The final web application will allow users to:

- Select a User ID
- Choose a recommendation algorithm
- Generate Top Movie Recommendations
- Display predicted ratings
- Provide a clean and interactive user interface

The Streamlit application will utilize the trained recommendation models developed throughout this project.

In [1]:
import pandas as pd
import numpy as np
import torch
import pickle
import os

print("Libraries Loaded Successfully!")

Libraries Loaded Successfully!


# Load Dataset

We load the MovieLens dataset along with movie titles that will be displayed inside the Streamlit application.

In [2]:
ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t",
    names=["user_id","movie_id","rating","timestamp"]
)

movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0,1],
    names=["movie_id","movie_title"]
)

ratings = ratings.merge(
    movies,
    on="movie_id"
)

print("Dataset Loaded Successfully!")

ratings.head()

Dataset Loaded Successfully!


,user_id,movie_id,rating,timestamp,movie_title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


# Load Trained Neural Collaborative Filtering Model

The trained NCF model will be used to generate personalized movie recommendations.

In [3]:
class NCF(torch.nn.Module):

    def __init__(self,
                 num_users,
                 num_movies,
                 embedding_dim=50):

        super().__init__()

        self.user_embedding = torch.nn.Embedding(
            num_users,
            embedding_dim
        )

        self.movie_embedding = torch.nn.Embedding(
            num_movies,
            embedding_dim
        )

        self.fc1 = torch.nn.Linear(
            embedding_dim*2,
            128
        )

        self.fc2 = torch.nn.Linear(
            128,
            64
        )

        self.output = torch.nn.Linear(
            64,
            1
        )

        self.relu = torch.nn.ReLU()

        self.dropout = torch.nn.Dropout(0.2)

    def forward(self,user,movie):

        user_vec = self.user_embedding(user)
        movie_vec = self.movie_embedding(movie)

        x = torch.cat(
            [user_vec,movie_vec],
            dim=1
        )

        x = self.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.relu(self.fc2(x))

        return self.output(x)

In [4]:
num_users = ratings.user_id.nunique()
num_movies = ratings.movie_id.nunique()

model = NCF(
    num_users,
    num_movies
)

model.load_state_dict(
    torch.load(
        "models/ncf_model.pth",
        map_location=torch.device("cpu")
    )
)

model.eval()

print("NCF Model Loaded Successfully!")

NCF Model Loaded Successfully!


# Create User and Movie Encoders

The recommendation model uses encoded IDs internally.

We recreate these mappings for inference in the Streamlit application.

In [5]:
user_to_encoded = {
    user:i
    for i,user in enumerate(
        sorted(ratings.user_id.unique())
    )
}

movie_to_encoded = {
    movie:i
    for i,movie in enumerate(
        sorted(ratings.movie_id.unique())
    )
}

encoded_to_movie = {
    v:k
    for k,v in movie_to_encoded.items()
}

print("Mappings Created Successfully!")

print("Users :",len(user_to_encoded))
print("Movies:",len(movie_to_encoded))

Mappings Created Successfully!
Users : 943
Movies: 1682


# Notebook Summary

In this notebook, we successfully:

- Loaded the MovieLens dataset
- Loaded the trained NCF model
- Recreated user and movie encodings
- Prepared all components required for building the Streamlit web application

The next notebook will implement the complete interactive Streamlit interface.